In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

In [ ]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from mlp_model.losses.bce_loss import BCELoss
from mlp_model.losses.cce_loss import CCELoss
from mlp_model.losses.mse_loss import MSELoss
from mlp_model.activations.relu import ReLU
from mlp_model.activations.tanh import Tanh
from mlp_model.activations.sigmoid import Sigmoid
import visualisation.classification_visualise as cl_vs
import classification_training_methods as cl_train

In [ ]:
data_kind = "simple" # simple, three_gauss
data_count = 1000 # 100, 500, 1000, 10000

train_filepath = Path(project_root) / "data/classification/train" / f"data.{data_kind}.train.{data_count}.csv"
test_filepath = Path(project_root) / "data/classification/test" / f"data.{data_kind}.test.{data_count}.csv"

In [ ]:
df = pd.read_csv(train_filepath)
df.head(5)

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
X = df.drop(["cls"], axis=1).to_numpy(dtype=float)
y = (df["cls"] - 1).to_numpy().reshape(-1, 1)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, 
    stratify=y.reshape(-1), random_state=42)

In [ ]:
plt.figure(figsize=(8, 6))

for cls in np.unique(y):
    mask = y_train.reshape(-1) == cls

    plt.scatter(
        X_train[mask, 0],
        X_train[mask, 1],
        label=f"Class {cls}",
        alpha=0.7,
        s=20
    )

In [ ]:
df_test = pd.read_csv(test_filepath)

X_test = df_test.drop(["cls"], axis=1).to_numpy(dtype=float)
y_test = (df_test["cls"] - 1).to_numpy().reshape(-1, 1)

## Porównanie klasyfikatora liniowego z siecią głęboką

In [ ]:
trainings_lin = cl_train.run_training(hidden_sizes=(), X_train=X_train, y_train=y_train, X_val=X_val,
                                        y_val=y_val, epochs=500, activation_hidden=ReLU(), learning_rate=0.01,
                                        loss_cls=BCELoss(), classes_count=2)

cl_vs.plot_decision_boundary(trainings_lin[0]["trainer"], X_val, y_val, "Linear")
cl_vs.plot_weights(trainings_lin[0]["history"], layer_idx=0, n_weights=2, type="Linear")

trainings_lin_dict = {"Linear": trainings_lin}

cl_vs.plot_loss(trainings_lin_dict, "train_loss")
cl_vs.plot_loss(trainings_lin_dict, "val_loss")

display(cl_train.summarize_results(trainings_lin_dict))

In [ ]:
test_results = cl_train.evaluate_runs(trainings_lin, X_test, y_test)
print("Linear test results: ")
display(test_results)
print("Linear average test results: ")
display(test_results[["test_loss", "accuracy", "precision", "f1_score"]
].agg(["mean", "std"]))

In [ ]:
trainings_deep = cl_train.run_training(hidden_sizes=(16,), X_train=X_train, y_train=y_train, X_val=X_val,
                                        y_val=y_val, activation_hidden=ReLU(), epochs=500, learning_rate=0.01,
                                        loss_cls=BCELoss(), classes_count=2)

cl_vs.plot_decision_boundary(trainings_deep[0]["trainer"], X_val, y_val, "Deep")
cl_vs.plot_weights(trainings_deep[0]["history"], layer_idx=1, n_weights=16, type="Deep")

trainings_deep_dict = {"Linear": trainings_deep}

cl_vs.plot_loss(trainings_deep_dict, "train_loss")
cl_vs.plot_loss(trainings_deep_dict, "val_loss")

display(cl_train.summarize_results(trainings_deep_dict))

In [ ]:
test_results = cl_train.evaluate_runs(trainings_deep, X_test, y_test)
print("Deep test results: ")
display(test_results)
print("Deep average test results: ")
display(test_results[["test_loss", "accuracy", "precision", "f1_score"]
].agg(["mean", "std"]))